In [1]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import os
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 4782 - DL/HRM_Reconstruction/code"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)
from GPT2_Model.GPT2_Sudoku import GPT2_Baseline, BaselineConfig, TrainConfig



device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Mounted at /content/gdrive
Device set to cuda


In [2]:
#Load the hyperparameters, scheduler, etc... required for training
train_size = 2**10
test_size = 2**8
batch_size = 2**8

model_config = BaselineConfig(num_layers=8,num_heads=8, embedding_dim=512)
train_config = TrainConfig(batch_per_iter=batch_size,grad_acc_factor=1)
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = GPT2_Baseline(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')


In [3]:
from Datasets.Sudoku_DataLoader import get_loaders
train_loader, val_loader = get_loaders(train_size, test_size, batch_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.csv:   0%|          | 0.00/719M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Map:   0%|          | 0/1024 [00:00<?, ? examples/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

In [4]:

import torch

def train(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=10,
    scheduler=None,
    grad_clip=None,
):
    model.to(device)
    print(f"Trainable parameters: \
        {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    for epoch in range(epochs):
        # ---- TRAIN ----
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            # Adjust depending on your dataloader format
            x, y = batch
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            _, loss = model(x, y)

            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ---- VALIDATION ----
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                x, y = batch
                x = x.to(device)
                y = y.to(device)

                _, loss = model(x,y)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        # ---- SCHEDULER ----
        if scheduler is not None:
            scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
train(model, train_loader,val_loader, optimizer, device)

Trainable parameters:         25,361,408


W0425 19:26:07.343000 6939 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch 1/10 | Train Loss: 2.5295 | Val Loss: 2.5308
Epoch 2/10 | Train Loss: 2.5286 | Val Loss: 2.5308
Epoch 3/10 | Train Loss: 2.5290 | Val Loss: 2.5308
Epoch 4/10 | Train Loss: 2.5299 | Val Loss: 2.5308
Epoch 5/10 | Train Loss: 2.5293 | Val Loss: 2.5308
Epoch 6/10 | Train Loss: 2.5296 | Val Loss: 2.5308
Epoch 7/10 | Train Loss: 2.5290 | Val Loss: 2.5308
Epoch 8/10 | Train Loss: 2.5294 | Val Loss: 2.5308
Epoch 9/10 | Train Loss: 2.5286 | Val Loss: 2.5308
Epoch 10/10 | Train Loss: 2.5285 | Val Loss: 2.5308


In [5]:

#Test how the model infers after training
def print_sudoku_comparison(x, pred, y):
    x = x.view(9, 9)
    pred = pred.view(9, 9)
    y = y.view(9, 9)

    RED = "\033[91m"
    RESET = "\033[0m"

    print("Input / Prediction (errors marked)")

    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("-" * 33)

        row = ""
        for j in range(9):
            if j % 3 == 0 and j != 0:
                row += "| "

            if x[i, j] != 0:
                # given clue
                val = str(x[i, j].item())
            else:
                if pred[i, j] == y[i, j]:
                    val = str(pred[i, j].item())
                else:
                    # color incorrect prediction red
                    val = f"{RED}{pred[i, j].item()}{RESET}"

            row += val + " "

        print(row)
    print()

import torch
def infer_all_samples(model, val_loader, device):
    model.eval()
    model.to(device)

    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            x = x.to(device)
            y = y.to(device)

            logits, loss = model(x, y)
            preds = logits.argmax(dim=-1)

            for i in range(x.size(0)):
                print_sudoku_comparison(
                    x[i].cpu(),
                    preds[i].cpu(),
                    y[i].cpu()
                )
infer_all_samples(model, val_loader, device)

Input / Prediction (errors marked)
0 9 0 | 8 0 1 | 2 6 8 
0 3 8 | 8 2 8 | 4 0 6 
8 6 8 | 0 8 1 | 8 8 8 
---------------------------------
8 7 8 | 3 0 0 | 1 4 8 
0 2 0 | 8 5 8 | 7 8 8 
8 1 3 | 3 8 8 | 0 8 2 
---------------------------------
8 1 5 | 9 3 0 | 8 8 8 
8 8 8 | 7 1 5 | 0 8 4 
9 8 7 | 2 8 6 | 0 8 4 

Input / Prediction (errors marked)
0 9 0 | 8 0 0 | 8 6 3 
0 8 8 | 8 8 8 | 9 0 0 
8 6 8 | 7 8 3 | 2 0 8 
---------------------------------
8 8 4 | 3 0 5 | 8 8 8 
5 2 0 | 8 8 8 | 6 3 8 
8 1 0 | 3 8 9 | 0 5 8 
---------------------------------
8 8 5 | 8 1 0 | 8 8 4 
1 8 8 | 0 1 7 | 8 2 4 
9 2 3 | 9 8 8 | 0 7 0 

Input / Prediction (errors marked)
0 8 0 | 7 0 8 | 8 8 1 
0 6 8 | 9 8 8 | 2 0 0 
8 4 8 | 5 2 1 | 9 8 8 
---------------------------------
4 8 8 | 6 0 0 | 8 8 8 
0 9 0 | 8 4 5 | 0 8 6 
8 1 0 | 8 9 2 | 4 8 8 
---------------------------------
9 8 5 | 8 3 0 | 8 7 8 
8 8 8 | 0 1 8 | 0 3 4 
3 5 1 | 2 8 8 | 6 8 0 

Input / Prediction (errors marked)
5 6 1 | 7 3 9 | 8 9 8 
0 8 8 | 8